# Detecting Spam Email with Bayes

Bayes' theorem is one of the fundamental ideas in probability theory and machine learning. In this notebook, we will first explain what Bayes' theorem means, and then use it to build a simple spam email classifier from scratch in Python.

Bayesian thinking is a way of updating our belief when new evidence appears. For example, imagine you can't find your keys. At first, you may believe they are equally likely to be in your backpack, on your desk, or in your pocket. After checking your backpack and finding nothing, you become less confident that the keys are in the backpac and more confident that they are somewhere else.

A spam classifier works in a similar way. Before reading an email, we may have a general belief about how likely it is to be spam. After observing words such as **"free"**, **"winner"**, or a suspicious link, we update that belief.

In this notebook, we will use two case studies:

1. A simple one-feature classifier using the word **"free"**
2. A slightly more realistic Naive Bayes classifier using three signals:
   - **"free"**
   - **"winner"**
   - **a suspicious link**


## 💙 1. Bayes' Theorem

Mathematically, Bayes' theorem is written as:

$$
P(A \mid B)=\frac{P(B \mid A)P(A)}{P(B)}
$$

where:

* **P(A)** is called the **prior probability**: our initial belief about event (A) before observing evidence.
* **P(B | A)** is the **likelihood**: the probability of observing evidence (B) if event (A) is true.
* **P(B)** is the **evidence** or **marginal probability**: the overall probability of observing evidence (B) regardless (A).
* **P(A | B)** is the **posterior probability**: the updated probability of event (A) after observing evidence (B).

In a spam classifier, this means:

$$
P(\text{spam} \mid \text{evidence})
=
\frac{
P(\text{evidence} \mid \text{spam})P(\text{spam})
}{
P(\text{evidence})
}
$$

The relationship can be summarized as:

> **Posterior = Likelihood × Prior / Evidence**

Bayes' theorem allows us to update an initial belief after observing new information. This may sound a bit abstract, let's see what each item means through real problem.


## 💙 2. Case Study 1: Detecting Spam Emails with the Word "free"

Suppose you receive an email saying:

> "Congratulations, you get a free gift from *En Språk Resa* for your hard work in building your Swedish vocabulary."

You are happy, but you are not sure whether the email is real or spam.

To make a simple decision, we use historical email data. The history statistics is as followed:

In [2]:
import pandas as pd
data = [["Spam", 200, 120],["Not spam", 800, 40],["Total", 1000, 160]]
df = pd.DataFrame(data, columns=["Email type", "Type total", 'Containing "free"'])
df

,Email type,Type total,"Containing ""free"""
0,Spam,200,120
1,Not spam,800,40
2,Total,1000,160


### Calculating the Probabilities Needed in Bayes' Theorem

In this first case study, we only use one signal:

> Does the email contain the word **"free"**?

Therefore:

- A = the email is spam
- B = the email contains the word **"free"**

So the Bayes Theorem becomes

$$
P(\text{spam} \mid \text{containing free})
=
\frac{
P(\text{containing free} \mid \text{spam})P(\text{spam})
}{
P(\text{containing free})
}
$$

We will calculate the four parts of Bayes' theorem step by step:

1. Prior probability: P(A)
2. Likelihood: P(B|A)
3. Evidence: P(B)
4. Posterior probability: P(A|B)


### 🩵 2.1 Prior Probability: P(A)

The prior probability is our belief **before reading the content of the email**.

From the historical data:

- Spam emails = 200
- Total emails = 1000

Therefore,

$$
P(A)=P(\text{spam})=\frac{200}{1000}=0.20
$$

This means that before observing any word in the email, we believe there is a **20% chance** that a randomly received email is spam.


In [3]:
spam_emails_count = df.loc[df["Email type"] == "Spam", 'Type total'].values[0]
total_emails_count = df.loc[df["Email type"] == "Total", "Type total"].values[0]
P_a = spam_emails_count / total_emails_count
print(f"P(A) = {P_a:.2%}")

P(A) = 20.00%


### 🩵 2.2 Likelihood: P(B|A)

The likelihood asks:

> If an email is spam, how likely is it to contain the word **"free"**?

From the historical data:

- Spam emails containing **"free"** = 120
- Total spam emails = 200

Therefore,

$$
P(B|A)=P(\text{free}|\text{spam})
=
\frac{120}{200}
=
0.60
$$

This means that **60%** of spam emails contain the word **"free"**.


In [4]:
spam_containing_free_count = df.loc[df["Email type"] == "Spam", 'Containing "free"'].values[0]
P_b_given_a = spam_containing_free_count / spam_emails_count
print(f"P(B|A) = {P_b_given_a:.2%}")

P(B|A) = 60.00%


### 🩵 2.3 Evidence: P(B)

The evidence is the overall probability of observing the word **"free"**, regardless of whether the email is spam or not.

From the whole dataset:

- Emails containing **"free"** = 160
- Total emails = 1000

Therefore,

$$
P(B)=P(\text{free})=\frac{160}{1000}=0.16
$$

This means that **16%** of all emails contain the word **"free"**.


In [5]:
all_containing_free_count = df.loc[df["Email type"] == "Total", 'Containing "free"'].values[0]
P_b = all_containing_free_count / total_emails_count
print(f"P(B) = {P_b:.2%}")

P(B) = 16.00%


### 🩵 2.4 Posterior Probability: P(A|B)

Now we calculate the updated probability that the email is spam after observing that it contains the word **"free"**.

$$
P(A|B)
=
\frac{P(B|A)P(A)}{P(B)}
$$

Substituting the numbers:

$$
P(\text{spam}|\text{free})
=
\frac{P(\text{free}|\text{spam}) \times P(\text{spam})}{P(\text{free})}
=
\frac{0.60\times0.20}{0.16}
=
0.75
$$

Therefore,

> If a new email contains the word **"free"**, there is a **75% probability** that it is spam.


In [6]:
P_a_given_b = P_b_given_a * P_a / P_b
print(f"P(A|B) = {P_a_given_b:.2%}")

P(A|B) = 75.00%


### Interpretation of Case Study 1

Before reading the email, the probability that it was spam was only:

$$
P(\text{spam})=20\%
$$

After observing the word **"free"** in the new email, the probability increased to:

$$
P(\text{spam}|\text{free})=75\%
$$

This is the core idea of Bayes' theorem:

> New evidence updates our belief.

This simple example uses only one word. In real spam detection, we usually use many signals together. That leads us to **Case Study 2**.


---


## 💙 3. Case Study 2: A Spam Email Classifier with Multiple Signals

Now suppose we receive another email:

> "You are a winner! Click this link to get your free gift. https://www.ensprakresa.com/"

This email contains three suspicious signals:

- the word **"free"**
- the word **"winner"**
- a suspicious link

Now our question is:

> What is the probability that this email is spam?

To answer this, we use a simple **Naive Bayes classifier** with the help of the statistical data we collected before.


In [11]:
df["Containing \"winner\""] = [80, 10, 90]
df["Containing a link"] = [150, 100, 250]
df

,Email type,Type total,"Containing ""free""","Containing ""winner""",Containing a link
0,Spam,200,120,80,150
1,Not spam,800,40,10,100
2,Total,1000,160,90,250


### 🩵 3.1 Prior probability: P(A)

Before reading the content of a new email, we only know the overall proportion of spam emails in our historical dataset.

From the training data:

- Spam emails = 200
- Total emails = 1000

Therefore,

$$
P(A) = P(\text{spam})=\frac{200}{1000}=0.20
$$

This means that before observing any evidence, we believe there is a **20% chance** that a randomly received email is spam. This step is exactly the same as in 2.1.

In [12]:
spam_emails_count = df.loc[df["Email type"] == "Spam", 'Type total'].values[0]
total_emails_count = df.loc[df["Email type"] == "Total", "Type total"].values[0]
P_spam = spam_emails_count / total_emails_count
print(f"P(A) = {P_spam:.2%}")

P(A) = 20.00%


### 🩵 3.2. Likelihood: P(B|A)

Instead of one piece of evidence as in Case Study 1, we now have **three pieces of evidence**. We treat them as one combined event:

$$
B=\{\text{free},\text{winner},\text{link}\}
$$

Our goal is still to compute

$$
P(\text{spam}|B).
$$

The likelihood asks:

> If the email is spam, how likely is it to contain all three features?

From the historical dataset:

$$
P(\text{free}|\text{spam})
=
\frac{120}{200}
=
0.60
$$

$$
P(\text{winner}|\text{spam})
=
\frac{80}{200}
=
0.40
$$

$$
P(\text{link}|\text{spam})
=
\frac{150}{200}
=
0.75
$$


Using the **Naive Bayes assumption**, we assume that the features are **conditionally independent**.

This means that **after we already know whether the email is Spam or Not Spam**, the presence of one feature does not affect the probability of another feature.

What does this mean?

Suppose we already know that an email **is spam** (we know the email class = given the email class).

Under this condition, we assume that

- whether the email contains **"free"**
- whether it contains **"winner"**
- whether it contains a **suspicious link**

do not influence one another.

Therefore,

$$
P(B|\text{spam})
=
P(\text{free}|\text{spam})
\times
P(\text{winner}|\text{spam})
\times
P(\text{link}|\text{spam})
$$

$$
P(B|\text{spam})
=
0.60\times0.40\times0.75
=
0.18
$$

In [13]:
P_free_given_spam = df.loc[df["Email type"] == "Spam", 'Containing "free"'].values[0] / spam_emails_count
P_winner_given_spam = df.loc[df["Email type"] == "Spam", 'Containing "winner"'].values[0] / spam_emails_count
P_link_given_spam = df.loc[df["Email type"] == "Spam", 'Containing a link'].values[0] / spam_emails_count
P_B_given_spam = P_free_given_spam * P_winner_given_spam * P_link_given_spam
print(f"P(B|A) = {P_B_given_spam:.2%}")

P(B|A) = 18.00%


### Why is it called **Naive**?

The assumption above is **not completely true** in real life.

For example, in spam emails,

- an email containing **"winner"** is also more likely to contain **"free"**,
- and both are more likely to appear together with a suspicious link.

These features are therefore **correlated** rather than perfectly independent.

However, Naive Bayes **pretends** that once we know whether an email is spam, these features become independent. This simplification makes the computation much easier while often producing surprisingly accurate results.

Machine learning researchers are actually quite honest with their naming. They basically said: "**This assumption is pretty naive... but let's see if it works.**"

Notice that we are **not** saying

> "free" and "winner" are independent.

Instead, we are saying

> **Once we already know the email is spam**, knowing that it contains **"free"** does not change the probability that it also contains **"winner"**.

This extra condition ("given the email is spam") is why the assumption is called **conditional independence**.

 However, let's continue to calculate the likelihood for non-spam emails for the convenience of the next step..

For not spam emails:

$$
P(\text{free}|\text{not spam})
=
\frac{40}{800}
=
0.05
$$

$$
P(\text{winner}|\text{not spam})
=
\frac{10}{800}
=
0.0125
$$

$$
P(\text{link}|\text{not spam})
=
\frac{100}{800}
=
0.125
$$

Likewise,

$$
P(B|\text{not spam})
=
P(\text{free}|\text{not spam})
\times
P(\text{winner}|\text{not spam})
\times
P(\text{link}|\text{not spam})
$$

$$
P(B|\text{not spam})
=
0.05\times0.0125\times0.125
=
0.000078125
$$

> The email pattern **"free + winner + link"** is much more common in spam emails than in normal emails.

In [14]:
nonspam_emails_count = df.loc[df["Email type"] == "Not spam", 'Type total'].values[0]
P_free_given_nonspam = df.loc[df["Email type"] == "Not spam", 'Containing "free"'].values[0] / nonspam_emails_count
P_winner_given_nonspam = df.loc[df["Email type"] == "Not spam", 'Containing "winner"'].values[0] / nonspam_emails_count
P_link_given_nonspam = df.loc[df["Email type"] == "Not spam", 'Containing a link'].values[0] / nonspam_emails_count
P_B_given_nonspam = P_free_given_nonspam * P_winner_given_nonspam * P_link_given_nonspam
P_nonspam = nonspam_emails_count / total_emails_count
print(f"P(B|not A) = {P_B_given_nonspam:.7%}")

P(B|not A) = 0.0078125%


### 🩵 3.3 Evidence: P(B)

The evidence is the overall probability of observing all the three features together:

- contains **"free"**
- contains **"winner"**
- contains a suspicious link

regardless of whether the email is spam or not.

In other words:

> If we randomly pick one email from the entire dataset, what is the probability that it contains all three features?

At this point, you might wonder:

> Why can't I simply calculate

$$
P(B)=P(\text{free})P(\text{winner})P(\text{link})
$$

or

$$
P(B) = 1- P(\text{non-free})P(\text{non-winner})P(\text{non-link})
$$
However, this would assume that **free**, **winner**, and **link** are independent across all emails. That is usually not true. In real spam emails, these features are often correlated: an email containing **"winner"** is also more likely to contain **"free"** and a suspicious link.


#### How to calculate P(B) when B contains multiple factors
Instead, we calculate the evidence P(B) by considering both possible classes in this example:

- Spam
- Not Spam

Since every email must be either **Spam** or **Not Spam**:

```text
                All Emails
               /          \
          Spam          Not Spam
           |                |
        Has B           Has B
```

Therefore, the total probability of observing the evidence is:

$$
P(B)
=
P(B\cap \text{spam})
+
P(B\cap \text{not spam})
$$

Using the conditional probability identity

$$
P(A\cap B)=P(B|A)P(A) = P(A|B)P(B),
$$

Even though we can also write

$$
P(B)
=
P(\text{spam}|B)P(B)
+
P(\text{not spam}|B)P(B)
$$

but each items in this equation is not easy to calculate. So we use:

$$
P(B)
=
P(B|\text{spam})P(\text{spam})
+
P(B|\text{not spam})P(\text{not spam})
$$

This is **not another application of Bayes' theorem**. It is the **Law of Total Probability**. We are simply adding the contribution from each possible class.

From the previous steps:

$$
P(B|\text{spam})=0.18
$$

$$
P(B|\text{not spam})=0.000078125
$$

and

$$
P(\text{spam})=0.20
$$

$$
P(\text{not spam})=0.80
$$

Therefore,

$$
P(B)
=
0.18\times0.20
+
0.000078125\times0.80
$$

$$
P(B)=0.0360625
$$

This means that approximately **3.61% of all incoming emails** contain all three features simultaneously.

> The evidence P(B) works as the normalizing denominator in Bayes' theorem. It makes sure that the posterior probabilities across all classes sum to 1.

In [15]:
P_B = P_B_given_spam * P_spam + P_B_given_nonspam * (1 - P_a)
print(f"P(B) = {P_B:.7%}")

P(B) = 3.6062500%


### 🩵 3.4. Posterior Probability: P(A|B)

Finally, Bayes' theorem updates our belief.

$$
P(\text{spam}|B)
=
\frac{
P(B|\text{spam})P(\text{spam})
}{
P(B)
}
$$

Substituting the numbers:

$$
P(\text{spam}|B)
=
\frac{0.18\times0.20}
{0.0360625}
$$

$$
P(\text{spam}|B)
=
0.9983
$$

Therefore,

> There is approximately a **99.83% probability** that this email is spam.


In [16]:
P_spam_given_B = P_B_given_spam * P_spam / P_B
print(f"P(spam|B) = {P_spam_given_B:.7%}")

P(spam|B) = 99.8266898%


---

### Interpretation

Initially, before reading the email:

$$
P(\text{spam})=20\%
$$

After observing the three signals:

- **"free"**
- **"winner"**
- suspicious link

the probability becomes:

$$
P(\text{spam}|B)=99.83\%
$$

This is much higher than in Case Study 1, because the classifier has received more evidence.

## 💙 4. Final Summary

In Case Study 1, we used only one signal: the word **"free"**.

- Before reading the email, the spam probability was **20%**.
- After observing **"free"**, the spam probability increased to **75%**.

In Case Study 2, we used three signals:

- **"free"**
- **"winner"**
- a suspicious link

With more evidence, the probability increased to approximately **99.83%**.

This notebook demonstrates how Bayes' theorem updates our belief step by step. As we observe more evidence, the posterior probability changes accordingly.

This simple idea forms the foundation of many machine learning algorithms, including the Naive Bayes classifier.


## 💙 Appendix: Where Does Bayes' Theorem Come From?

In order to understand the Bayes Theorem better, let's recall the definition of **conditional probability**.

The probability of event \(A\) given that event \(B\) has already occurred is

$$
P(A|B)=\frac{P(A\cap B)}{P(B)}.
$$

Rearranging the equation gives

$$
P(A\cap B)=P(A|B)P(B).
$$

Likewise, we can reverse the roles of \(A\) and \(B\):

$$
P(B|A)=\frac{P(A\cap B)}{P(A)},
$$

which can also be rearranged as

$$
P(A\cap B)=P(B|A)P(A).
$$

Notice that both equations describe the **same joint probability** $P(A \cap B)$. Therefore,

$$
P(A|B)P(B)
=
P(B|A)P(A).
$$

Finally, dividing both sides by \(P(B)\) gives

$$
P(A|B)
=
\frac{P(B|A)P(A)}{P(B)},
$$

which is exactly **Bayes' theorem**.

---

### Key idea

Bayes' theorem is **not a formula that needs to be memorized**.

Instead, it is derived directly from the definition of conditional probability.